In [1]:
from ultralytics import YOLO
import cv2
import subprocess
import time
from collections import defaultdict

In [2]:
model = YOLO("best.pt")
def preprocess_frame(frame):
    resized = cv2.resize(frame, (1280, 720))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    return gray_3ch
def speak(text):
    subprocess.Popen(["espeak", text])
last_spoken_time = 0
last_crosswalk_spoken_time = 0
last_feedback_message = ""
cooldown = 3
crosswalk_cooldown = 10
position_cooldown = 10

In [5]:
input_path = "vid1.mp4"
output_path = "output_video.mp4"

In [6]:
cap = cv2.VideoCapture(input_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = 1280
height = 720

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
class_position_last_spoken = defaultdict(float)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = preprocess_frame(frame)
    result = model(frame, conf=0.50, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)
    detected_classes = []
    pedestrian_positions = []
    pedestrian_count_by_position = defaultdict(int)  # count per position

    # ---------- PROCESS DETECTIONS ----------
    for box, conf, cls in zip(boxes, confs, classes):
        x1, y1, x2, y2 = map(int, box)
        class_name = model.names[cls]
        detected_classes.append(class_name)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Determine horizontal position
        center_x = (x1 + x2) // 2
        if center_x < width // 3:
            position = "left"
        elif center_x < 2 * width // 3:
            position = "center"
        else:
            position = "right"
        # Count pedestrians per position
        if class_name == "pedestrians":
            pedestrian_positions.append(position)
            pedestrian_count_by_position[position] += 1

    # -------------------- DECISION ENGINE --------------------
    feedback_message = ""
    current_time = time.time()

    def position_allowed(class_name, position):
        """Check if this class+position combo is off cooldown."""
        return current_time - class_position_last_spoken[(class_name, position)] > position_cooldown

    def get_position_for_class(target_class):
        """Find the position of the first detected instance of a class."""
        for box, cls in zip(boxes, classes):
            if model.names[cls] == target_class:
                x1, y1, x2, y2 = map(int, box)
                center_x = (x1 + x2) // 2
                if center_x < width // 3:
                    return "left"
                elif center_x < 2 * width // 3:
                    return "center"
                else:
                    return "right"
        return None

    if current_time - last_spoken_time > cooldown:
        # --------- CROWD DETECTION ---------
        # Find position with more than 4 pedestrians
        crowd_position = None
        for pos, count in pedestrian_count_by_position.items():
            if count > 4:
                crowd_position = pos
                break

        if crowd_position and position_allowed("crowd", crowd_position):
            feedback_message = f"crowd detected on {crowd_position}"

        # --------- CROSSWALK + SIGNAL ---------
        elif "crosswalk" in detected_classes and "signal" in detected_classes:
            if current_time - last_crosswalk_spoken_time > crosswalk_cooldown:
                position = get_position_for_class("crosswalk")
                if position and position_allowed("crosswalk", position):
                    feedback_message = "crosswalk ahead good to cross"

        elif "crosswalk" in detected_classes:
            if current_time - last_crosswalk_spoken_time > crosswalk_cooldown:
                position = get_position_for_class("crosswalk")
                if position and position_allowed("crosswalk", position):
                    feedback_message = "crosswalk ahead"

        # --------- OBSTACLE DETECTION ---------
        elif "obstacle" in detected_classes:
            position = get_position_for_class("obstacle")
            if position and position_allowed("obstacle", position):
                feedback_message = f"obstacle detected on {position}"

        # --------- VEHICLE DETECTION ---------
        elif "vehicle" in detected_classes:
            position = get_position_for_class("vehicle")
            if position and position_allowed("vehicle", position):
                feedback_message = f"vehicle detected on {position}"

        # --------- STAIRS DETECTION ---------
        elif "stairs" in detected_classes:
            position = get_position_for_class("stairs")
            if position and position_allowed("stairs", position):
                feedback_message = f"stairs detected on {position}"

        # --------- PEDESTRIAN DETECTION (less than crowd) ---------
        elif "pedestrians" in detected_classes:
            if pedestrian_positions:
                dominant_position = max(set(pedestrian_positions),
                                        key=pedestrian_positions.count)
                if position_allowed("pedestrians", dominant_position):
                    feedback_message = f"pedestrians detected on {dominant_position}"

        # Speak if message exists
        if feedback_message:
            speak(feedback_message)
            last_spoken_time = current_time
            last_feedback_message = feedback_message
            if "crosswalk" in feedback_message:
                last_crosswalk_spoken_time = current_time
            # Update per class+position cooldown
            for pos in ["left", "center", "right"]:
                if pos in feedback_message:
                    detected_class = feedback_message.split(" detected")[0].split("crowd")[0].strip()
                    if "crowd" in feedback_message:
                        class_position_last_spoken[("crowd", pos)] = current_time
                    elif "crosswalk" in feedback_message:
                        class_position_last_spoken[("crosswalk", pos)] = current_time
                    elif "obstacle" in feedback_message:
                        class_position_last_spoken[("obstacle", pos)] = current_time
                    elif "vehicle" in feedback_message:
                        class_position_last_spoken[("vehicle", pos)] = current_time
                    elif "stairs" in feedback_message:
                        class_position_last_spoken[("stairs", pos)] = current_time
                    elif "pedestrians" in feedback_message:
                        class_position_last_spoken[("pedestrians", pos)] = current_time
                    break

    # -------------------- ALWAYS DISPLAY LAST FEEDBACK AT BOTTOM --------------------
    if last_feedback_message:
        cv2.rectangle(frame, (0, height - 60), (width, height), (0, 0, 0), -1)
        cv2.putText(frame,
                    last_feedback_message,
                    (30, height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA)

    # Write frame
    out.write(frame)

# -------------------- CLEANUP --------------------
cap.release()
out.release()
print("Processing complete with audio and visual feedback.")

Processing complete with audio and visual feedback.
